# Análises para Dashboard Executivo

## Objetivo

Analisar os arquivos tratados na camada silver e identificar os principais riscos, ativos prioritários e oportunidades de melhoria no processo de correção.

A análise busca responder às seguintes questões da liderança:

- Qual é o nível atual de exposição ao risco?
- Quais ativos demandam maior atenção?
- Onde devem ser priorizados os esforços de correção?
- O processo atual de tratamento de vulnerabilidades está sendo efetivo?

## 01. Preparação dos dados


Os dados utilizados nesta análise foram previamente tratados e validados na camada Silver.

In [50]:
import pandas as pd

ativos = pd.read_parquet('../data/silver/ativos.parquet')
vulnerabilidades = pd.read_parquet('../data/silver/vulnerabilidades.parquet')

## 02. Pergunta 1 — Exposição atual ao risco


### Volume de vulnerabilidades

In [51]:
total_vulnerabilidades = vulnerabilidades.shape[0]
print(f'Total de vulnerabilidades: {total_vulnerabilidades}')

Total de vulnerabilidades: 1995


### Distribuição de vulnerabilidades por severidade

In [52]:
vulnerabilidades['severidade'].value_counts(dropna=False)

severidade
Média      505
Crítica    499
Alta       488
Baixa      477
NaN         26
Name: count, dtype: int64

### Distribuição de vulnerabilidades pro status

In [53]:
vulnerabilidades['status'].value_counts(dropna=False)

status
Em tratamento    525
Aceita           516
Aberta           488
Corrigida        466
Name: count, dtype: int64

### Severidade × Status

In [54]:
pd.crosstab(vulnerabilidades['severidade'],vulnerabilidades['status'])

status,Aberta,Aceita,Corrigida,Em tratamento
severidade,,,,
Alta,122,117,125,124
Baixa,124,131,100,122
Crítica,123,131,112,133
Média,114,134,120,137


### Percentual de vulnerabilidades corrigidas

In [55]:
percentual_corrigidas = (pd.crosstab(vulnerabilidades['severidade'],vulnerabilidades['status'],normalize='index')['Corrigida'] * 100).round(2)

percentual_corrigidas

severidade
Alta       25.61
Baixa      20.96
Crítica    22.44
Média      23.76
Name: Corrigida, dtype: float64

### Insight

A distribuição por severidade permite dimensionar o nível de risco presente na base, enquanto o cruzamento entre severidade e status permite identificar quanto desse risco permanece pendente de tratamento.

Para análises de priorização, vulnerabilidades com status Aberta ou Em tratamento são consideradas pendentes. Vulnerabilidades Aceitas não são tratadas como backlog de correção, pois representam riscos aceitos.

## 03. Pergunta 2 — Ativos que demandam atenção


### Isolando vulnerabilidades que ainda exigem correção

In [56]:
vulnerabilidades_pendentes = vulnerabilidades[(vulnerabilidades['status'].isin(['Aberta', 'Em tratamento']))]

print(pd.crosstab(vulnerabilidades_pendentes['severidade'],vulnerabilidades_pendentes['status']))

print(f'Total de vulnerabilidades pendentes: {vulnerabilidades_pendentes.shape[0]}')

status      Aberta  Em tratamento
severidade                       
Alta           122            124
Baixa          124            122
Crítica        123            133
Média          114            137
Total de vulnerabilidades pendentes: 1013


### Identificando as vulnerabilidades com maior pontos de atenção (Crítica e Alta) que estão pendentes

In [57]:
vulnerabilidades_risco = vulnerabilidades_pendentes[vulnerabilidades_pendentes['severidade'].isin(['Crítica', 'Alta'])]
vulnerabilidades_risco

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
1,V00002,A0060,Alta,Qualys,2026-06-01,NaT,Em tratamento,7.5
2,V00003,A0080,Alta,Veracode,2026-08-18,NaT,Aberta,1.7
3,V00004,A0086,Alta,Qualys,2026-06-16,NaT,Aberta,2.5
4,V00005,A0089,Alta,Cloud Security,2026-02-12,NaT,Em tratamento,4.4
5,V00006,A0025,Crítica,Cloud Security,2026-05-02,NaT,Aberta,8.9
...,...,...,...,...,...,...,...,...
1984,V01990,A0014,Alta,Veracode,2026-09-18,NaT,Aberta,1.2
1988,V01994,A0124,Crítica,Cloud Security,2026-02-05,NaT,Em tratamento,4.2
1989,V01995,A0086,Crítica,Qualys,2026-09-06,NaT,Em tratamento,5.8
1993,V01999,A0046,Crítica,Veracode,2026-07-26,NaT,Aberta,9.0


### Ativos com maior concentração de vulnerabilidades prioritárias

In [58]:
vulnerabilidades_risco.groupby('ativo_id').size().sort_values(ascending=False).head(10)

ativo_id
A0086    8
A0059    7
A0067    7
A0161    7
A0120    7
A0133    6
A0025    6
A0183    6
A0178    6
A0024    6
dtype: int64

### Insight

Os ativos com maior concentração de vulnerabilidades Críticas e Altas em aberto ou em tratamento representam os principais pontos de atenção do ambiente.

A concentração dessas vulnerabilidades em determinados ativos permite direcionar os esforços de análise e correção para os ambientes mais expostos, em vez de distribuir os recursos de forma uniforme entre todos os ativos.

Dessa forma, o ranking dos ativos fornece uma visão objetiva de onde o risco está mais concentrado.

## 04. Pergunta 3 — Priorização dos esforços de correção


### Presença de vulnerabilidades Críticas e Altas


In [59]:
vulnerabilidades_altas_criticas = vulnerabilidades[vulnerabilidades['severidade'].isin(['Alta', 'Crítica'])].shape[0]
print(f'Total de vulnerabilidades altas e críticas: {vulnerabilidades_altas_criticas}')

percentual_altas_criticas = ((vulnerabilidades_altas_criticas / total_vulnerabilidades) * 100).__round__(2)
print(f'Percentual de vulnerabilidades altas e críticas: {percentual_altas_criticas:}%')

Total de vulnerabilidades altas e críticas: 987
Percentual de vulnerabilidades altas e críticas: 49.47%


### Tempo de correção de cada vulnerabilidade

In [60]:
vulnerabilidades_corrigidas = vulnerabilidades[vulnerabilidades['status'] == 'Corrigida'].copy()
vulnerabilidades_corrigidas['tempo_correcao'] = (vulnerabilidades_corrigidas['data_correcao'] - vulnerabilidades_corrigidas['data_abertura'])
vulnerabilidades_corrigidas

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score,tempo_correcao
12,V00013,A0060,Baixa,Pentest,2026-02-09,2026-03-14,Corrigida,2.7,33 days
21,V00022,A0046,Alta,Cloud Security,2026-04-02,2026-04-29,Corrigida,7.6,27 days
22,V00023,A0200,NaN,Veracode,2026-08-08,2026-09-26,Corrigida,2.7,49 days
29,V00030,A0145,Baixa,Qualys,2026-05-13,2026-05-31,Corrigida,2.4,18 days
36,V00037,A0009,Alta,Pentest,2026-06-03,2026-08-03,Corrigida,1.5,61 days
...,...,...,...,...,...,...,...,...,...
1966,V01972,A0074,Crítica,Veracode,2026-01-02,2026-03-31,Corrigida,8.4,88 days
1969,V01975,A0067,Alta,Veracode,2026-01-03,2026-01-05,Corrigida,9.7,2 days
1971,V01977,A0012,Baixa,Cloud Security,2026-08-04,2026-10-06,Corrigida,0.2,63 days
1979,V01985,A0020,Alta,Pentest,2026-03-11,2026-05-16,Corrigida,8.4,66 days


### Métricas de média de dias para correção, tempo minimo e máximo

In [61]:
vulnerabilidades_corrigidas['tempo_correcao'].describe()

count                        456
mean     48 days 02:44:12.631578
std      30 days 13:51:12.944286
min              1 days 00:00:00
25%             24 days 00:00:00
50%             47 days 00:00:00
75%             70 days 06:00:00
max            230 days 00:00:00
Name: tempo_correcao, dtype: object

### Mediana por severidade

In [62]:
vulnerabilidades_corrigidas.groupby('severidade')['tempo_correcao'].median()

severidade
Alta      47 days
Baixa     46 days
Crítica   46 days
Média     51 days
Name: tempo_correcao, dtype: timedelta64[us]

### Insight

Os tempos medianos de correção apresentam valores próximos entre as diferentes severidades.

Esse comportamento merece atenção, pois vulnerabilidades Críticas e Altas não apresentam um tempo de correção significativamente inferior às vulnerabilidades de menor severidade.

## 05. Pergunta 4 — Efetividade do processo


### Periodo de abertura das vulnerabilidades

In [63]:
vulnerabilidades['data_abertura'].min(), vulnerabilidades['data_abertura'].max()

(Timestamp('2026-01-01 00:00:00'), Timestamp('2026-12-09 00:00:00'))

### Vulnerabilidades registradas por mês

In [64]:
vulnerabilidades['mes_abertura'] = vulnerabilidades['data_abertura'].dt.to_period('M')
vulnerabilidades_abertas_mes = vulnerabilidades['mes_abertura'].value_counts().sort_index()
vulnerabilidades_abertas_mes

mes_abertura
2026-01    292
2026-02    213
2026-03    217
2026-04    193
2026-05    232
2026-06    217
2026-07    236
2026-08    234
2026-09    136
2026-10      2
2026-12      1
Freq: M, Name: count, dtype: int64

### Vulnerabilidades corrigidas por mês

In [65]:
vulnerabilidades['mes_correcao'] = vulnerabilidades['data_correcao'].dt.to_period('M')
vulnerabilidades_corrigidas_mes = vulnerabilidades['mes_correcao'].value_counts().sort_index()
vulnerabilidades_corrigidas_mes

mes_correcao
2026-01     8
2026-02    29
2026-03    51
2026-04    52
2026-05    47
2026-06    50
2026-07    55
2026-08    48
2026-09    45
2026-10    49
2026-11    27
2026-12     3
Freq: M, Name: count, dtype: int64

### Insight

Ao longo do período analisado, o volume de novas vulnerabilidades registradas foi superior ao volume de vulnerabilidades corrigidas na maior parte dos meses.

Esse comportamento indica que a correção atual não está acompanhando suficientemente a entrada de novas vulnerabilidades. Além disso, o tempo mediano de correção é de 47 dias e apresenta pouca diferença entre as severidades.

Portanto, o processo possui capacidade de tratamento, mas precisa aumentar sua capacidade de correção e direcionar os esforços de forma mais consistente para vulnerabilidades Críticas e Altas.

## 06. Conclusões

**Exposição ao risco**
- A base possui 1.995 vulnerabilidades, distribuídas entre diferentes níveis de severidade e status de tratamento. 
- Atualmente, 1.013 vulnerabilidades estão Abertas ou Em tratamento, representando o principal volume de risco que ainda demanda ação.
- O nível de exposição deve ser analisado principalmente pela combinação entre severidade e status.
- Vulnerabilidades Críticas e Altas pendentes representam a maior prioridade de risco.

**Ativos que demandam maior atenção**
- Os ativos com vulnerabilidades Críticas e Altas demandam maior atenção.
- O ranking desses ativos permite direcionar os esforços para os ambientes que concentram maior exposição ao risco, considerando também suas características de sistema, criticidade, domínio e localidade.


**Priorização de esforços**
- O tempo mediano geral de correção é de 47 dias.
- Os tempos medianos são semelhantes entre as severidades.
- Vulnerabilidades Críticas e Altas não apresentam um tempo de correção significativamente menor.

**Efetividade**
- Os dados indicam que o processo não está sendo suficientemente efetivo para acompanhar a demanda.
- Ao longo do período analisado, o volume de novas vulnerabilidades registradas foi muito maior do que o volume de vulnerabilidades corrigidas. Além disso, o tempo mediano de correção é de 47 dias e não apresenta diferença relevante entre as severidades.
- Existe a oportunidade de aumentar a capacidade de tratamento e melhorar a priorização de vulnerabilidades de maior severidade .


### Recomendações

1. Priorizar vulnerabilidades Críticas e Altas nos ativos com maior concentração de risco.
2. Avaliar SLAs diferenciados de correção conforme a severidade.
3. Investigar por que vulnerabilidades de maior severidade apresentam tempos de correção semelhantes aos demais níveis.
4. Monitorar continuamente novas vulnerabilidades, correções e backlog.

## 07. Geração da camada Gold

Após a análise dos dados, os principais datasets utilizados no dashboard são preparados e disponibilizados na camada Gold.

A camada Gold contém dados enriquecidos e organizados para consumo analítico, mantendo o tratamento realizado na Silver e adicionando atributos necessários para as análises de negócio.

Realizando um left join de vulnerabilidades e ativos, para consolidação em um data frame único.

In [69]:
vulnerabilidades_gold = vulnerabilidades.merge(ativos,on='ativo_id',how='left')

Criando colunas que irão ajudar na criação e visualização das métricas no dashboard.

In [70]:
vulnerabilidades_gold['tempo_correcao_dias'] = (
    vulnerabilidades_gold['data_correcao'] -
    vulnerabilidades_gold['data_abertura']
).dt.days

Excluindo colunas que não serão utilizadas

In [73]:
vulnerabilidades_gold = vulnerabilidades_gold[
    [
        'vuln_id',
        'ativo_id',
        'severidade',
        'origem',
        'criticidade',
        'dominio',
        'ambiente',
        'localidade',
        'data_abertura',
        'data_correcao',
        'status',
        'cvss_score',
        'tempo_correcao_dias'
    ]
]

vulnerabilidades_gold.head()

,vuln_id,ativo_id,severidade,origem,criticidade,dominio,ambiente,localidade,data_abertura,data_correcao,status,cvss_score,tempo_correcao_dias
0,V00001,A0078,Baixa,Cloud Security,Crítica,Dominio_7,Homologação,São Paulo,2026-03-21,NaT,Em tratamento,2.8,NaN
1,V00002,A0060,Alta,Qualys,Média,Dominio_1,NaN,Curitiba,2026-06-01,NaT,Em tratamento,7.5,NaN
2,V00003,A0080,Alta,Veracode,Crítica,Dominio_9,Produção,Recife,2026-08-18,NaT,Aberta,1.7,NaN
3,V00004,A0086,Alta,Qualys,Média,Dominio_3,Produção,Recife,2026-06-16,NaT,Aberta,2.5,NaN
4,V00005,A0089,Alta,Cloud Security,Média,Dominio_6,Produção,Recife,2026-02-12,NaT,Em tratamento,4.4,NaN


Salvando os data frames na camada gold no formato CSV para consumo do dashboard.

In [75]:
vulnerabilidades_gold.to_csv(
    '../data/gold/vulnerabilidades.csv',
    index=False
)